# 可选实验：Model Representation

<figure>
  <img src="./images/C1_W1_L3_S1_Lecture_b.png" alt="Model representation 课程图示" style="width:600px;max-width:100%;height:auto;">
</figure>


## 目标

在本实验中，你将：

- 学习实现单变量 linear regression（线性回归）模型 $f_{w,b}$。


## 符号说明

下面汇总了本实验中使用的符号。

| 符号 | 说明 | Python（如适用） |
| :--- | :--- | :--- |
| $a$ | scalar（标量），不使用粗体 | |
| $\mathbf{a}$ | vector（向量），使用粗体 | |
| **Regression** | | |
| $\mathbf{x}$ | 训练样本的 feature 值，本实验为面积（1000 sqft） | `x_train` |
| $\mathbf{y}$ | 训练样本的 target，本实验为价格（千美元） | `y_train` |
| $x^{(i)}$, $y^{(i)}$ | 第 $i$ 个训练样本 | `x_i`, `y_i` |
| $m$ | 训练样本数 | `m` |
| $w$ | 模型参数：weight（权重） | `w` |
| $b$ | 模型参数：bias（偏置） | `b` |
| $f_{w,b}(x^{(i)})$ | 参数为 $w,b$ 时，模型对 $x^{(i)}$ 的预测：$f_{w,b}(x^{(i)})=wx^{(i)}+b$ | `f_wb` |


## 工具

本实验将使用：

- NumPy：常用的科学计算库。
- Matplotlib：常用的数据绘图库。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')

## 问题描述

<img align="left" src="./images/C1_W1_L3_S1_trainingdata.png" alt="房价预测的训练数据" style="width:380px;padding:10px;">

与课程视频一样，本实验以房价预测为例。数据集只有两个数据点：面积为 1000 平方英尺（sqft）的房屋售价为 300,000 美元，面积为 2000 sqft 的房屋售价为 500,000 美元。这两个点构成我们的 *training set（训练集）*。本实验的面积单位为 1000 sqft，价格单位为千美元。

| 面积（1000 sqft） | 价格（千美元） |
| --- | --- |
| 1.0 | 300 |
| 2.0 | 500 |

你将拟合一个经过这两个点的 linear regression 模型（如图中的蓝色直线），用来预测其他房屋的价格，例如一套面积为 1200 sqft 的房屋。


请运行下面的代码单元，创建变量 `x_train` 和 `y_train`。数据存储在一维 NumPy array 中。


In [ ]:
# x_train is the input variable (size in 1000 square feet)
# y_train is the target (price in 1000s of dollars)
x_train = np.array([1.0, 2.0])
y_train = np.array([300.0, 500.0])
print(f"x_train = {x_train}")
print(f"y_train = {y_train}")

> **注意：** 本课程打印输出时会经常使用 Python 的 f-string 格式，详见 [Python 文档](https://docs.python.org/3/tutorial/inputoutput.html)。生成输出时，花括号中的表达式会被求值。


### 训练样本数 `m`

我们用 `m` 表示训练样本数。NumPy array 的 `.shape` 属性返回一个 Python tuple，其中每个元素对应一个维度的长度。对于这里的一维 array，`x_train.shape[0]` 就是 array 的长度，也就是样本数，如下所示。


In [ ]:
# m is the number of training examples
print(f"x_train.shape: {x_train.shape}")
m = x_train.shape[0]
print(f"Number of training examples is: {m}")

也可以使用 Python 的 `len()` 函数，如下所示。


In [ ]:
# m is the number of training examples
m = len(x_train)
print(f"Number of training examples is: {m}")

### 训练样本 `x_i, y_i`

我们用 $(x^{(i)},y^{(i)})$ 表示第 $i$ 个训练样本。Python 的 index 从 0 开始，因此 $(x^{(0)},y^{(0)})=(1.0,300.0)$，$(x^{(1)},y^{(1)})=(2.0,500.0)$。

要访问 NumPy array 中的某个值，可以使用对应的 index。例如，`x_train[0]` 访问 `x_train` 中 index 为 0 的元素。运行下面的代码，获取第 $i$ 个训练样本。


In [ ]:
i = 0 # Change this to 1 to see (x^1, y^1)

x_i = x_train[i]
y_i = y_train[i]
print(f"(x^({i}), y^({i})) = ({x_i}, {y_i})")

### 绘制数据


可以使用 Matplotlib 的 `scatter()` 函数绘制这两个点，如下面的代码所示。

- 参数 `marker` 和 `c` 将数据点设置为红色叉号；默认样式为蓝色圆点。

还可以使用 Matplotlib 的其他函数设置图表标题和坐标轴标签。


In [ ]:
# Plot the data points
plt.scatter(x_train, y_train, marker='x', c='r')
# Set the title
plt.title("Housing Prices")
# Set the y-axis label
plt.ylabel('Price (in 1000s of dollars)')
# Set the x-axis label
plt.xlabel('Size (1000 sqft)')
plt.show()

## 模型函数

<img align="left" src="./images/C1_W1_L3_S1_model.png" alt="Linear regression 模型" style="width:380px;padding:10px;">

如课程所述，linear regression 的模型函数将 `x` 映射为 `y`，表示为：

$$
f_{w,b}(x^{(i)})=wx^{(i)}+b \tag{1}
$$

这个公式表示一条直线，不同的 $w$ 和 $b$ 会在图中产生不同的直线。

下面通过代码加深理解，先设置 $w=100$、$b=100$。

**注意：你可以返回这个单元，调整模型参数 `w` 和 `b`。**


In [ ]:
w = 100
b = 100
print(f"w: {w}")
print(f"b: {b}")

下面计算两个数据点对应的 $f_{w,b}(x^{(i)})$。可以分别写出：

- 对于 $x^{(0)}$：`f_wb = w * x[0] + b`。
- 对于 $x^{(1)}$：`f_wb = w * x[1] + b`。

当数据点较多时，逐个编写会十分冗长且重复。因此，可以像下面的 `compute_model_output` 函数一样，通过 `for` loop 计算模型输出。

> **注意：** 参数说明中的 `ndarray (m,)` 表示 shape 为 `(m,)` 的 NumPy array；`scalar` 表示只有数值大小、没有 array 维度的参数。
>
> **注意：** `np.zeros(n)` 返回一个包含 $n$ 个零的一维 NumPy array。


In [ ]:
def compute_model_output(x, w, b):
    """
    Computes the prediction of a linear model
    Args:
      x (ndarray (m,)): Data, m examples 
      w,b (scalar)    : model parameters  
    Returns
      f_wb (ndarray (m,)): model prediction
    """
    m = x.shape[0]
    f_wb = np.zeros(m)
    for i in range(m):
        f_wb[i] = w * x[i] + b
        
    return f_wb

下面调用 `compute_model_output` 函数，并绘制输出结果。


In [ ]:
tmp_f_wb = compute_model_output(x_train, w, b,)

# Plot our model prediction
plt.plot(x_train, tmp_f_wb, c='b',label='Our Prediction')

# Plot the data points
plt.scatter(x_train, y_train, marker='x', c='r',label='Actual Values')

# Set the title
plt.title("Housing Prices")
# Set the y-axis label
plt.ylabel('Price (in 1000s of dollars)')
# Set the x-axis label
plt.xlabel('Size (1000 sqft)')
plt.legend()
plt.show()

可以看到，设置 $w=100$、$b=100$ 得到的直线没有很好地拟合数据。

### 挑战

尝试不同的 $w$ 和 $b$。要让直线拟合这两个数据点，参数应该取什么值？

#### 提示

点击下面绿色的“提示”，查看如何选择 $w$ 和 $b$。


<details>
<summary><span style="color:darkgreen;"><strong>提示</strong></span></summary>

尝试设置 $w=200$、$b=100$。

</details>


### Prediction

现在已有模型，可以完成最初的预测任务：估计一套面积为 1200 sqft 的房屋价格。由于 $x$ 的单位是 1000 sqft，因此输入为 $x=1.2$。


In [ ]:
w = 200                         
b = 100    
x_i = 1.2
cost_1200sqft = w * x_i + b    

print(f"${cost_1200sqft:.0f} thousand dollars")

## 恭喜完成实验！

在本实验中，你了解到：

- Linear regression 构建了 feature（特征）与 target（目标值）之间的关系模型。
- 上面的例子中，feature 是房屋面积，target 是房屋价格。
- Simple linear regression 有两个参数 $w$ 和 $b$，通过 *training data* 拟合它们的值。
- 确定模型参数后，就可以对新数据进行预测。
